In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
os.chdir("..")

In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN, THINK_START_TOKEN



os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [4]:
tokenizer = initialize_tokenizer(model_id)

In [5]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name, model_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/{model_name}/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [6]:
original_dataset = load_dataset(
    "dmitriihook/qwq-32b-planning-mystery-2-24k"
)["train"]

In [7]:
task_name = "plan_generation_po"
eval_results = [
    load_dataset_from_file("blocksworld_mystery_2", task_name, model_name)["instances"] for model_name in [
        "qwq-32b-steered-full", "qwq-32b-original-full"
    ]
]
eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]

In [11]:
def collect_correct_ids(eval_results: dict) -> list[int]:
    clean_ids = []
    for idx in range(48):
        if eval_results[idx]["llm_correct"]:
            clean_ids.append(idx)

    return clean_ids

correct_ids =[
    collect_correct_ids(eval_results[0]),
    collect_correct_ids(eval_results[1])
]

In [12]:
import json

file_path = "steered_generations_2_full.json"
dataset = json.load(open(file_path, "r"))

In [13]:
set(correct_ids[0]) - set(correct_ids[1])

{8, 10, 18, 29, 39}

In [16]:
print(dataset[0]["original_input"])

Okay, let's tackle this problem step by step. First, I need to understand the initial conditions and the goal clearly. The initial conditions are:

- Aura on Block A, B, D
- Essence on A, C, D
- Block C harmonizes Block B
- Nexus is present

The goal is to have:

1. Block A harmonizes Block D
2. Block B harmonizes Block C
3. Block D harmonizes Block B

Hmm, so the goal requires three harmonization relationships. Let me note that the existing harmonization is C harmonizes B. The other two need to be created. 

Looking at the actions available: Illuminate, Distill, Silence, Divest. Each has specific preconditions and effects. Let me recall their details.

First, the actions:

1. **Illuminate object**:
   - Requires: Essence object, Aura object, Nexus.
   - After: Pulse object becomes true, Essence, Aura, Nexus become false.

2. **Silence object**:
   - Requires: Pulse object.
   - After: Essence, Aura, Nexus become true again, Pulse becomes false.

3. **Distill object from another object

In [ ]:
print(dataset[8]["original_input"])

Okay, let's tackle this problem step by step. First, I need to understand the initial conditions and the goal clearly. 

The initial conditions are:
- Aura Block A
- Aura Block B
- Essence Block B
- Essence Block C
- Block C harmonizes Block D
- Block D harmonizes Block A
- Nexus

The goal is to have:
- Block B harmonizes Block C
- Block C harmonizes Block A
- Block D harmonizes Block B

Hmm, so I need to figure out a sequence of actions (Illuminate, Distill, Silence, Divest) that will lead from the initial state to the goal state, following all the given rules. Let me recall the actions and their preconditions and effects.

First, let me list out the actions again to make sure I have them right:

1. **Illuminate object**:
   - Requires: Essence object, Aura object, Nexus.
   - After: Pulse object becomes true. Essence object, Aura object, Nexus become false.

2. **Silence object**:
   - Requires: Pulse object.
   - After: Essence object, Aura object, Nexus become true. Pulse becomes f

: 

In [18]:
print(dataset[8]["steered_generation"])

user
I am playing with a set of objects. Here are the actions I can do

   Illuminate object
   Distill object from another object
   Silence object
   Divest object from another object

I have the following restrictions on my actions:
    To perform Illuminate action, the following facts need to be true: Essence object, Aura object, Nexus.
    Once Illuminate action is performed the following facts will be true: Pulse object.
    Once Illuminate action is performed the following facts will be false: Essence object, Aura object, Nexus.
    To perform Silence action, the following facts need to be true: Pulse object.
    Once Silence action is performed the following facts will be true: Essence object, Aura object, Nexus.    
    Once Silence action is performed the following facts will be false: Pulse object.
    To perform Distill action, the following needs to be true: Essence other object, Pulse object.
    Once Distill action is performed the following will be true: Nexus, Essence 

In [12]:
steered_batched = []

for path in Path("intermediate_results_4_full_b").glob("*.json"):
    with open(path, "r") as f:
        data = json.load(f)
        steered_batched += data

steered_batched = {
    x["idx"]: x for x in steered_batched
}

In [37]:
steered_batched.keys()

dict_keys([12, 13, 14, 15, 36, 37, 38, 39])

In [53]:
gen_1 = steered_batched[12]["steered_generation"].split("<think>\n")[1]
gen_2 = dataset[12]["steered_generation"].split("<think>\n")[1]
orig = dataset[12]["original_input"]

common_prefix = ""
for i in range(len(gen_1)):
    if gen_1[i] == orig[i]:
        common_prefix += gen_1[i]
    else:
        break
print(len(common_prefix))

6636


In [63]:
row = original_dataset[12]

In [66]:
pref = "\n\n".join(row["generation"].split("\n\n")[:40])

In [67]:
tokenize_blocksworld_generation(tokenizer, original_dataset[12], pref)

tensor([[151644,    872,    198,  ...,     25, 151645,    198]])

In [54]:
gen_1

'Okay, let\'s see. I need to solve this problem where I have certain initial conditions and a goal. The actions available are Swim, Respond, Fire, and Deduct, each with their own preconditions and effects. Let me start by understanding the problem step by step.\n\nFirst, the initial conditions are:\n- Craving\n- Fever on Blocks A, C, D\n- Marble on Blocks A, B, D\n- Block B mines Block C.\n\nThe goal is to have:\n- Block A mines Block B\n- Block B mines Block D\n- Block C mines Block A.\n\nHmm, so I need to figure out a sequence of actions that will lead from the initial state to the goal state. Let me recall the actions and their effects again to make sure I have them right.\n\n**Actions and their details:**\n\n1. **Swim object:**\n   - Requires: Marble on the object, Fever on the object, and Craving.\n   - After doing it: \n     - Shadow on the object becomes true.\n     - Marble, Fever, and Craving on the object become false.\n\n2. **Fire object:**\n   - Requires: Shadow on the obje

In [55]:
print(
    gen_1[len(common_prefix):],
)

 then Shadow on A becomes true, and Marble, Fever, and Craving (the global) become false. Wait, but Craving is a global fact. Wait, the Swim action's effects say "once Swim is performed, the following will be false: Marble object, Fever object, Craving." So, after swimming, the global Craving is set to false. That might be a problem because some actions require Craving to be true.

Alternatively, if I Swim D, same thing: D's Marble and Fever go away, and Craving becomes false. So, perhaps we need to be careful with when we use Swim, because it removes Craving, which is needed for some actions.

Alternatively, maybe we can use Fire to get back the Fever and Marble and Craving. Because Fire requires Shadow on the object. So, after swimming, the object has Shadow, so we can fire it again to get back the Marble, Fever, and Craving (but the global Craving would be set back to true? Wait, the Fire action's effects say that after Fire, the object's Marble, Fever, and Craving become true. Wait

In [56]:
print(
    gen_2[len(common_prefix):],
)

 then Shadow on A becomes true, and Marble, Fever, and Craving (the global) become false. Wait, but Craving is a global fact. Wait, the Swim action's effects say "once Swim is performed, the following will be false: Marble object, Fever object, Craving." So, after swimming, the global Craving is set to false. That might be a problem because some actions require Craving to be true.

Alternatively, if I Swim D, same thing: D's Marble and Fever go away, and Craving becomes false. So, perhaps we need to be careful with when we use Swim, because it removes Craving, which is needed for some actions.

Alternatively, maybe we can use Fire to get back the Fever and Marble and Craving. Because Fire requires Shadow on the object. So, after swimming, the object has Shadow, so we can fire it again to get back the Marble, Fever, and Craving (but the global Craving would be set back to true? Wait, the Fire action's effects say that after Fire, the object's Marble, Fever, and Craving become true. Wait

In [14]:
print(dataset[12]["original_input"])

Okay, let's see. I need to solve this problem where I have certain initial conditions and a goal. The actions available are Swim, Respond, Fire, and Deduct, each with their own preconditions and effects. Let me start by understanding the problem step by step.

First, the initial conditions are:
- Craving
- Fever on Blocks A, C, D
- Marble on Blocks A, B, D
- Block B mines Block C.

The goal is to have:
- Block A mines Block B
- Block B mines Block D
- Block C mines Block A.

Hmm, so I need to figure out a sequence of actions that will lead from the initial state to the goal state. Let me recall the actions and their effects again to make sure I have them right.

**Actions and their details:**

1. **Swim object:**
   - Requires: Marble on the object, Fever on the object, and Craving.
   - After doing it: 
     - Shadow on the object becomes true.
     - Marble, Fever, and Craving on the object become false.

2. **Fire object:**
   - Requires: Shadow on the object.
   - After doing it:
 

In [9]:
print(dataset[16]["original_input"])

Okay, let's see. I need to solve this problem where the initial conditions are given, and I have to come up with a plan using the actions provided to reach the goal. Let me start by understanding the problem step by step.

First, the initial conditions are:
- Block A craves Block B (so Object Craves other object for A is B)
- Block C craves Block D (Object Craves other object for C is D)
- Harmony exists (Harmony fact is true)
- Planet Block B and Planet Block D are present (so Planet is true for B and D)
- Province Block A and Province Block C are true (so both A and C are in province)

The goal is to have:
- Block A craves Block C
- Block C craves Block D (this is already true, so maybe that's just maintained)
- Block D craves Block B (so D needs to crave B now, which isn't the case initially since C craves D, but D's craving isn't mentioned. Wait, the initial says C craves D, but D's craving isn't specified. The goal requires D to crave B, so that's a new condition.)

Wait, the init

In [42]:
print(dataset[16]["steered_generation"])

user
I am playing with a set of objects. Here are the actions I can do

   Attack object
   Feast object from another object
   Succumb object
   Overcome object from another object

I have the following restrictions on my actions:
    To perform Attack action, the following facts need to be true: Province object, Planet object, Harmony.
    Once Attack action is performed the following facts will be true: Pain object.
    Once Attack action is performed the following facts will be false: Province object, Planet object, Harmony.
    To perform Succumb action, the following facts need to be true: Pain object.
    Once Succumb action is performed the following facts will be true: Province object, Planet object, Harmony.    
    Once Succumb action is performed the following facts will be false: Pain object.
    To perform Overcome action, the following needs to be true: Province other object, Pain object.
    Once Overcome action is performed the following will be true: Harmony, Province